# Transfer Learning: Standing on the Shoulders of Pretrained Models

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/04-Transfer-Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/05-pytorch/'
for fn in ['q_top5.py', 'q_dinov2.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from IPython.display import display
device = "cuda" if torch.cuda.is_available() else "cpu"
import os
NUM_WORKERS = 2 if os.name != "nt" else 0   # Colab (Linux) uses 2 workers; set 0 when running locally on Windows

## Using models pretrained on ImageNet

`torchvision.models` provides many pretrained models. The new API selects weights with `weights=`, and the weights object also carries the preprocessing transforms and the class names.

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights).to(device).eval()
preprocess = weights.transforms()
preprocess

In [ ]:
from urllib.request import urlopen
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
img = Image.open(urlopen(url)).convert("RGB")
display(img.resize((256, int(256 * img.height / img.width))))

In [ ]:
with torch.no_grad():
    logits = model(preprocess(img)[None].to(device))
probs = logits.softmax(1)[0]
idx = probs.argmax().item()
weights.meta["categories"][idx], probs[idx].item()

## Q
Print the 5 most likely classes and their probabilities. (Hint: `torch.topk`)

Then try other images.

In [ ]:
# %load q_top5.py
%run -i q_top5.py

## timm

[timm (PyTorch Image Models)](https://github.com/huggingface/pytorch-image-models) collects over a thousand image models with pretrained weights, and it comes preinstalled on Colab.

In [ ]:
import timm
timm.list_models("convnext*", pretrained=True)[:10]

## Dataset: Oxford-IIIT Pet

37 cat and dog breeds, with only 3680 training images, about 100 per class. Training from scratch is hard, but easy with a pretrained model.

Here we download it from the Hugging Face Hub with 🤗 `datasets`.

In [ ]:
from datasets import load_dataset
pets = load_dataset("timm/oxford-iiit-pet")
pets

In [ ]:
class_names = pets["train"].features["label"].names
ex = pets["train"][0]
print(class_names[ex["label"]])
display(ex["image"].resize((200, 200)))

## Building the model

`timm.create_model(..., num_classes=37)` loads the pretrained weights and replaces the final classification layer with a new, randomly initialized 37-class one.

In [ ]:
model_name = "convnext_tiny.fb_in22k_ft_in1k"
model = timm.create_model(model_name, pretrained=True, num_classes=len(class_names))
data_cfg = timm.data.resolve_model_data_config(model)
train_tf = timm.data.create_transform(**data_cfg, is_training=True)
test_tf = timm.data.create_transform(**data_cfg, is_training=False)
print(data_cfg)
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
from torch.utils.data import DataLoader
def make_transform(tf):
    def fn(batch):
        return {"pixel_values": [tf(img.convert("RGB")) for img in batch["image"]], "label": batch["label"]}
    return fn
def collate(batch):
    return torch.stack([b["pixel_values"] for b in batch]), torch.tensor([b["label"] for b in batch])

train_ds = pets["train"].with_transform(make_transform(train_tf))
test_ds = pets["test"].with_transform(make_transform(test_tf))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=128, num_workers=NUM_WORKERS, collate_fn=collate)
xb, yb = next(iter(train_loader))
xb.shape, yb[:10]

In [ ]:
@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = 0
    for xb, yb in test_loader:
        with torch.autocast(device, dtype=torch.float16):
            correct += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item()
    return correct / len(test_ds)

def fit(model, params, epochs, lr):
    model.to(device)
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs * len(train_loader), pct_start=0.1)
    scaler = torch.amp.GradScaler(device)
    for epoch in range(epochs):
        model.train()
        t0 = time.perf_counter()
        for xb, yb in train_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            with torch.autocast(device, dtype=torch.float16):
                loss = F.cross_entropy(model(xb), yb, label_smoothing=0.1)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        print(f"epoch {epoch} loss={loss.item():.3f} test acc={evaluate(model):.4f} ({time.perf_counter()-t0:.0f}s)")

### Method 1: train only the last layer (linear probing)

Freeze the backbone (`requires_grad=False`) and train only the new classification layer. It is fast and less prone to overfitting.

In [ ]:
for p in model.parameters():
    p.requires_grad = False
head = model.get_classifier()
for p in head.parameters():
    p.requires_grad = True
print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
fit(model, head.parameters(), epochs=2, lr=1e-3)

### Method 2: fine-tune the whole model (fine-tuning)

Fine-tuning all parameters with a smaller learning rate usually improves things a bit more.

In [ ]:
for p in model.parameters():
    p.requires_grad = True
fit(model, model.parameters(), epochs=3, lr=1e-4)

In [ ]:
# Look at the predictions
model.eval()
for i in range(0, 3000, 500):
    ex = pets["test"][i]
    with torch.no_grad(), torch.autocast(device, dtype=torch.float16):
        pred = model(test_tf(ex["image"].convert("RGB"))[None].to(device)).argmax(1).item()
    display(ex["image"].resize((150, 150)))
    print("pred:", class_names[pred], " label:", class_names[ex["label"]])

## Q: the self-supervised DINOv2 model + classical machine learning

[DINOv2](https://huggingface.co/facebook/dinov2-small) is a Vision Transformer trained with self-supervised learning. It **never saw any labels**, yet its features are extremely useful.

* Load `facebook/dinov2-small` with `AutoImageProcessor` and `AutoModel` from `transformers`
* Take the `pooler_output` (the CLS token features) for each image
* Train a classifier with scikit-learn's `LogisticRegression` and check the test accuracy

(🤗 transformers is formally introduced in the next unit)

In [ ]:
# %load q_dinov2.py
%run -i q_dinov2.py